# No Arbitrage, Forwards, and Put-Call Parity

# Forwards

1. Continuous Compounding

Recall:

$$
e = \lim_{n \to \infty} \left(1+\frac{1}{n}\right)^n
$$

Also, if $f$ is continuous at

$$
\overline{L} = \lim_{x \to a} g(x),
$$

then

$$
\lim_{x \to a} f(g(x))
=
f\left(\lim_{x \to a} g(x)\right)
=
f(\overline{L}).
$$

Let $r>0$ be the annual interest rate, $n$ be the number of compounding periods per year, and $T$ be the time to maturity in years.

If \$1 is invested today, compounded $n$ times per year, then its value after $T$ years is

$$
\left(1+\frac{r}{n}\right)^{nT}.
$$

As $n \to \infty$,

$$
\begin{aligned}
\lim_{n \to \infty}
\left(1+\frac{r}{n}\right)^{nT}
&=
\lim_{n/r \to \infty}
\left[
\left(1+\frac{1}{n/r}\right)^{n/r}
\right]^{rT}
\\
&=
\left[
\lim_{n/r \to \infty}
\left(1+\frac{1}{n/r}\right)^{n/r}
\right]^{rT}
\\
&=
e^{rT}.
\end{aligned}
$$

Thus, under continuous compounding, \$1 grows to

$$
e^{rT}.
$$


2. Forwards

**Definition:** A forward contract is an over-the-counter instrument between two parties to buy or sell an asset at a specified future date $T$ for a predetermined delivery price $K$.

The terminal payoff of a long forward contract is

$$
S_T-K.
$$

Assuming no arbitrage, no dividends, and a continuously compounded risk-free interest rate $r$, construct a replicating portfolio:

- Borrow $Ke^{-rT}$.
- Buy one share of stock for $S_0$.

The initial cost of the portfolio is

$$
S_0-Ke^{-rT}.
$$

At maturity, the stock is worth $S_T$, while the loan has grown to

$$
Ke^{-rT}e^{rT}=K.
$$

Hence, the terminal payoff of the replicating portfolio is

$$
S_T-K,
$$

which is identical to the terminal payoff of the long forward contract.

Therefore, by the Law of One Price,

$$
V_0=S_0-Ke^{-rT}.
$$

For a newly initiated contract,

$$
V_0=0.
$$

Therefore,

$$
0=S_0-Ke^{-rT},
$$

and hence

$$
K=S_0e^{rT}.
$$

For an already-existing contract,

$$
V_0=S_0-Ke^{-rT}
$$

can be nonzero, where $K$ is predetermined.


3. Arbitrage

Consider newly initiated forward contracts.

If

$$
K<S_0e^{rT},
$$

today, we can:

- Short one share of stock to receive $S_0$.
- Invest $S_0$ at the risk-free rate.
- Enter a long forward contract to buy one share of stock at maturity for $K$.

At maturity, the investment has grown to

$$
S_0e^{rT}.
$$

We pay $K$ under the forward, receive one share, and use it to close the short position.

The risk-free profit is therefore

$$
S_0e^{rT}-K>0.
$$

Conversely, if

$$
K>S_0e^{rT},
$$

today, we can:

- Borrow $S_0$.
- Buy one share of stock.
- Enter a short forward contract to sell one share of stock at maturity for $K$.

At maturity, the loan has grown to

$$
S_0e^{rT}.
$$

We deliver the share through the forward and receive $K$.

The risk-free profit is therefore

$$
K-S_0e^{rT}>0.
$$

In [7]:
from math import exp

S0 = 100
r = 0.05
T = 1

def forward_delivery_price(S0, r, T):
    return S0 * exp(r * T)

def forward_arbitrage(S0, K_market, r, T, tol = 1e-10):
    K_fair = forward_delivery_price(S0, r, T)

    if K_fair > K_market + tol:
        return {
            "arbitrage": True,
            "strategy": ["short 1 stock", "invest S0", "long 1 forward"],
            "terminal_payoff": K_fair - K_market
        }

    elif K_fair < K_market - tol:
        return {
            "arbitrage": True,
            "strategy": ["borrow S0", "long 1 stock", "short 1 forward"],
            "terminal_payoff": K_market - K_fair
        }

    return {
        "arbitrage": False,
        "strategy": [],
        "terminal_payoff": 0
    }

K = forward_delivery_price(S0, r, T)
print(f"Forward price: {K:.4f}")
print(f"K_market: 110 => {forward_arbitrage(S0, 110, r, T)}")
print(f"K_market: 100 => {forward_arbitrage(S0, 100, r, T)}")
print(f"K_market: K_fair => {forward_arbitrage(S0, K, r, T)}")



Forward price: 105.1271
K_market: 110 => {'arbitrage': True, 'strategy': ['borrow S0', 'long 1 stock', 'short 1 forward'], 'terminal_payoff': 4.872890362397584}
K_market: 100 => {'arbitrage': True, 'strategy': ['short 1 stock', 'invest S0', 'long 1 forward'], 'terminal_payoff': 5.127109637602416}
K_market: K_fair => {'arbitrage': False, 'strategy': [], 'terminal_payoff': 0}


## Put-Call Parity

Consider two portfolios.

Portfolio A
- Long one European call
- Own a zero-coupon bond paying $K$ at maturity

Portfolio B
- Long one European put
- Long one share of stock

Their costs today are

$$
A_0 = C + Ke^{-rT},
\qquad
B_0 = P + S_0.
$$

At maturity, if

$$
S_T \leq K,
$$

then

$$
A_T = 0 + K = K,
$$

and

$$
B_T = (K-S_T)+S_T=K.
$$

Conversely, if

$$
S_T>K,
$$

then

$$
A_T=(S_T-K)+K=S_T,
$$

and

$$
B_T=0+S_T=S_T.
$$

Therefore, in both cases,

$$
A_T=B_T=\max(S_T,K).
$$

By the Law of One Price,

$$
A_0=B_0.
$$

Hence,

$$
C+Ke^{-rT}=P+S_0.
$$

Or equivalently,

$$
C-P=S_0-Ke^{-rT},
$$

which is the put-call parity.

In [8]:
def discount_factor(r, T):
    return exp(-r * T)

def pv_strike(K, r, T):
    return K * discount_factor(r, T)

def parity_rhs(S0, K, r, T):
    return S0 - pv_strike(K, r, T)

def parity_residual(C, P, S0, K, r, T):
    return C - P - parity_rhs(S0, K, r, T)

def call_payoff(ST, K):
    return max(ST-K, 0)

def put_payoff(ST, K):
    return max(K-ST, 0)

def portfolio_payoff(ST, K, call_qty=0, put_qty=0, stock_qty=0, bond_qty=0):
    return call_payoff(ST, K) * call_qty + put_payoff(ST, K) * put_qty + ST * stock_qty + K * bond_qty

def parity_arbitrage(C, P, S0, K, r, T, tol = 1e-10):
    R = parity_residual(C, P, S0, K, r, T)

    if R > tol:
        return {
            "arbitrage": True,
            "positions": {
                "call": -1,
                "put": +1,
                "stock": +1,
                "bond": -1
            },
            "strategy": ["short 1 call", "borrow PV(K)", "long 1 put", "long 1 stock"],
            "initial_profit": R
        }

    if R < -tol:
        return {
            "arbitrage": True,
            "positions": {
                "call": +1,
                "put": -1,
                "stock": -1,
                "bond": +1
            },
            "strategy": ["short 1 put", "short 1 stock", "long 1 call", "invest PV(K)"],
            "initial_profit": -R
        }

    return {
            "arbitrage": False,
            "positions": None,
            "strategy": [],
            "initial_profit": 0
        }

In [9]:
K = 100
C = 12
P = 5

print(f"C-P must equal: {parity_rhs(S0, K, r, T)}")

arbitrage_res = parity_arbitrage(C, P, S0, K, r, T)
arbitrage_pos =arbitrage_res['positions']
print(f"C = {C}, P = {P} => {arbitrage_res}")

for ST in [0, 25, 50, 75, 99, 100, 101, 125, 150, 175, 200]:
    A = portfolio_payoff(ST, K, call_qty=1, bond_qty=1)
    B = portfolio_payoff(ST, K, put_qty=1, stock_qty=1)
    arbitrage = portfolio_payoff(ST, K, call_qty=arbitrage_pos['call'], put_qty=arbitrage_pos['put'], stock_qty=arbitrage_pos['stock'], bond_qty=arbitrage_pos['bond'])

    print(f"ST = {ST:3}, A: {A}, B: {B}, arbitrage: {arbitrage}")

C-P must equal: 4.877057549928594
C = 12, P = 5 => {'arbitrage': True, 'positions': {'call': -1, 'put': 1, 'stock': 1, 'bond': -1}, 'strategy': ['short 1 call', 'borrow PV(K)', 'long 1 put', 'long 1 stock'], 'initial_profit': 2.122942450071406}
ST =   0, A: 100, B: 100, arbitrage: 0
ST =  25, A: 100, B: 100, arbitrage: 0
ST =  50, A: 100, B: 100, arbitrage: 0
ST =  75, A: 100, B: 100, arbitrage: 0
ST =  99, A: 100, B: 100, arbitrage: 0
ST = 100, A: 100, B: 100, arbitrage: 0
ST = 101, A: 101, B: 101, arbitrage: 0
ST = 125, A: 125, B: 125, arbitrage: 0
ST = 150, A: 150, B: 150, arbitrage: 0
ST = 175, A: 175, B: 175, arbitrage: 0
ST = 200, A: 200, B: 200, arbitrage: 0


## Experiments

1. Bounds

If $X_T \ge Y_T$ in all states, then $X_0 \ge Y_0$. Otherwise, suppose $X_0 < Y_0$. Today we can long $X$ and short $Y$, making a positive profit $Y_0 - X_0$. At time $T$, the net payoff is $X_T - Y_T \ge 0$. Arbitrage!

- Call bounds: $\max(S_0 - \mathrm{PV}(K), 0) \le C \le S_0$

$$
C_T = \max(S_T - K, 0) \le S_T
$$

for all states. Thus,

$$
C \le S_0.
$$

$$
C = S_0 - \mathrm{PV}(K) + P \ge S_0 - \mathrm{PV}(K).
$$

Also, $C \ge 0$. Hence,

$$
C \ge \max(S_0 - \mathrm{PV}(K), 0).
$$

- Put bounds: $\max(\mathrm{PV}(K)-S_0,0) \le P \le \mathrm{PV}(K)$

$$
P = C - S_0 + \mathrm{PV}(K) \le \mathrm{PV}(K)
$$

since $C \le S_0$.

Also,

$$
P = \mathrm{PV}(K) - S_0 + C \ge \mathrm{PV}(K) - S_0,
$$

and $P \ge 0$. Therefore,

$$
P \ge \max(\mathrm{PV}(K)-S_0,0).
$$

2. Quotes

Let

$$
x = S_0 - \mathrm{PV}(K).
$$

We can write the option premiums as

$$
C = \max(x,0)+s,
\qquad
P = \max(-x,0)+s,
$$

for some common slack variable $s$.

If $S_0 \ge \mathrm{PV}(K)$, then $x \ge 0$, so

$$
C = S_0-\mathrm{PV}(K)+s \le S_0
\Rightarrow
s \le \mathrm{PV}(K),
$$

and

$$
P=s\le \mathrm{PV}(K).
$$

If $S_0 < \mathrm{PV}(K)$, then $x<0$, so

$$
C=s\le S_0,
$$

and

$$
P=\mathrm{PV}(K)-S_0+s\le \mathrm{PV}(K)
\Rightarrow
s\le S_0.
$$

Thus,

$$
0\le s\le \min(S_0,\mathrm{PV}(K)).
$$

In [13]:
def generate_quotes(S0, K, r, T, slack_fraction=0.1):
    pvK = pv_strike(K, r, T)
    x = S0 - pvK

    slack_max = min(S0, pvK)
    slack = slack_max * slack_fraction

    C = max(x,0) + slack
    P = max(-x,0) + slack

    return C, P

def executable_parity_arbitrage(
    call_bid, call_ask,
    put_bid, put_ask,
    stock_bid, stock_ask,
    K, r, T, tol=1e-10
):
    pvK = pv_strike(K, r, T)

    # long call + invest + short put + short stock
    V0_1 = - call_ask - pvK + put_bid + stock_bid

    # long put + long stock + short call + borrow
    V0_2 = - put_ask - stock_ask + call_bid + pvK

    if V0_1 > tol:
        return {
            "arbitrage": True,
            "positions": {
                "call": +1,
                "put": -1,
                "stock": -1,
                "bond": +1
            },
            "strategy": ["short 1 put", "short 1 stock", "long 1 call", "invest PV(K)"],
            "initial_profit": V0_1
        }

    if V0_2 > tol:
        return {
            "arbitrage": True,
            "positions": {
                "call": -1,
                "put": +1,
                "stock": +1,
                "bond": -1
            },
            "strategy": ["short 1 call", "borrow PV(K)", "long 1 put", "long 1 stock"],
            "initial_profit": V0_2
        }

    return {
        "arbitrage": False,
        "positions": None,
        "strategy": [],
        "initial_profit": 0
    }

A parity violation in mid-prices is not necessarily an executable arbitrage once bid/ask spreads are included.

In [14]:
C, P = generate_quotes(S0, K, r, T)

cases = {
    "underpriced call": C - 2,
    "fair call": C,
    "overpriced call": C + 2
}

for name, C in cases.items():
    residual = parity_residual(C, P, S0, K, r, T)
    arbitrage_res = parity_arbitrage(C, P, S0, K, r, T)

    print(name)
    print(f"C: {C:.4f}, P: {P:.4f}, residual: {residual}")
    print(f"using mid-price: {arbitrage_res}")

    for gap in [0.2,0.4,0.6,0.8,1]:
        executable_arbitrage_res = executable_parity_arbitrage(
            C-gap, C+gap, P-gap, P+gap, S0-gap, S0+gap, K, r, T
        )
        print(f"using bid-ask spread = {gap*2}: {executable_arbitrage_res}")
    print()

underpriced call
C: 12.3894, P: 9.5123, residual: -2.0
using mid-price: {'arbitrage': True, 'positions': {'call': 1, 'put': -1, 'stock': -1, 'bond': 1}, 'strategy': ['short 1 put', 'short 1 stock', 'long 1 call', 'invest PV(K)'], 'initial_profit': 2.0}
using bid-ask spread = 0.4: {'arbitrage': True, 'positions': {'call': 1, 'put': -1, 'stock': -1, 'bond': 1}, 'strategy': ['short 1 put', 'short 1 stock', 'long 1 call', 'invest PV(K)'], 'initial_profit': 1.3999999999999915}
using bid-ask spread = 0.8: {'arbitrage': True, 'positions': {'call': 1, 'put': -1, 'stock': -1, 'bond': 1}, 'strategy': ['short 1 put', 'short 1 stock', 'long 1 call', 'invest PV(K)'], 'initial_profit': 0.799999999999983}
using bid-ask spread = 1.2: {'arbitrage': True, 'positions': {'call': 1, 'put': -1, 'stock': -1, 'bond': 1}, 'strategy': ['short 1 put', 'short 1 stock', 'long 1 call', 'invest PV(K)'], 'initial_profit': 0.20000000000001705}
using bid-ask spread = 1.6: {'arbitrage': False, 'positions': None, 'strate

In [16]:
for r_test in [0.01, 0.05, 0.1]:
    for K_test in [80, 100, 120]:
        for T_test in [0.25, 1, 2]:

            C, P = generate_quotes(S0, K_test, r_test, T_test)
            rhs = parity_rhs(S0, K_test, r_test, T_test)

            print(f"r={r_test:.2f}, K={K_test:3}, T={T_test:.2f}, C-P={C-P:.4f}, RHS={rhs:.4f}")

r=0.01, K= 80, T=0.25, C-P=20.1998, RHS=20.1998
r=0.01, K= 80, T=1.00, C-P=20.7960, RHS=20.7960
r=0.01, K= 80, T=2.00, C-P=21.5841, RHS=21.5841
r=0.01, K=100, T=0.25, C-P=0.2497, RHS=0.2497
r=0.01, K=100, T=1.00, C-P=0.9950, RHS=0.9950
r=0.01, K=100, T=2.00, C-P=1.9801, RHS=1.9801
r=0.01, K=120, T=0.25, C-P=-19.7004, RHS=-19.7004
r=0.01, K=120, T=1.00, C-P=-18.8060, RHS=-18.8060
r=0.01, K=120, T=2.00, C-P=-17.6238, RHS=-17.6238
r=0.05, K= 80, T=0.25, C-P=20.9938, RHS=20.9938
r=0.05, K= 80, T=1.00, C-P=23.9016, RHS=23.9016
r=0.05, K= 80, T=2.00, C-P=27.6130, RHS=27.6130
r=0.05, K=100, T=0.25, C-P=1.2422, RHS=1.2422
r=0.05, K=100, T=1.00, C-P=4.8771, RHS=4.8771
r=0.05, K=100, T=2.00, C-P=9.5163, RHS=9.5163
r=0.05, K=120, T=0.25, C-P=-18.5093, RHS=-18.5093
r=0.05, K=120, T=1.00, C-P=-14.1475, RHS=-14.1475
r=0.05, K=120, T=2.00, C-P=-8.5805, RHS=-8.5805
r=0.10, K= 80, T=0.25, C-P=21.9752, RHS=21.9752
r=0.10, K= 80, T=1.00, C-P=27.6130, RHS=27.6130
r=0.10, K= 80, T=2.00, C-P=34.5015, RHS=34